# 최종 파일

## 1. 데이터 크롤링
카드고릴라 카드탭에서 단종카드 제외 신용, 체크카드 크롤링.

### 1-1. 신용카드 

In [ ]:
# CRD 전체(806개) 크롤링: 리스트(검색) → 더보기 반복 → 상세 진입 → 혜택별 수집
# 저장 파일: cardgorilla_crd_806_fee.json

from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoAlertPresentException
import json, time, os


# 0) 설정

LIST_URL = "https://www.card-gorilla.com/search/card?cate=CRD&annual=0,5&pre=0,3&sort=ranking&is_discon=1"
OUT_PATH = "cardgorilla_crd_806_fee.json"

TARGET_N = 806            # 전체 카드 수(표시용). 실제 종료는 더보기 버튼 없어질 때까지.
SAVE_EVERY = 25           # 중간 저장 주기
SLEEP_CLICK_MORE = 1.2    # 더보기 클릭 후 대기
SLEEP_BETWEEN_CARDS = 0.5 # 카드 상세 페이지 간 대기

driver = wb.Chrome()
driver.implicitly_wait(3)
wait = WebDriverWait(driver, 12)


# 1) 팝업/새창 정리 유틸

def cleanup_popup(driver):
    # 새 창/탭 닫기
    if len(driver.window_handles) > 1:
        main = driver.window_handles[0]
        for h in driver.window_handles[1:]:
            driver.switch_to.window(h)
            driver.close()
        driver.switch_to.window(main)

    # alert 닫기
    try:
        driver.switch_to.alert.dismiss()
    except NoAlertPresentException:
        pass
    except:
        pass


# 2) 리스트에서 detail 링크 전부 수집

def collect_links_all():
    driver.get(LIST_URL)
    cleanup_popup(driver)

    # 카드 리스트가 로딩될 때까지
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.b_view[href^='/card/detail/']")))

    links = set()
    last_count = 0
    stagnant_round = 0

    while True:
        cleanup_popup(driver)

        # 현재 화면에 있는 detail 링크 수집
        elems = driver.find_elements(By.CSS_SELECTOR, "a.b_view[href^='/card/detail/']")
        for e in elems:
            href = e.get_attribute("href")
            if href:
                links.add(href)

        print(f"현재 링크 수: {len(links)} / {TARGET_N}(목표표시)")

        # 더보기 버튼 찾기 (없으면 종료)
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, "a.lst_more")
        except:
            print("✅ 더보기 버튼 없음 → 링크 수집 종료")
            break

        # 더보기 버튼 클릭
        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", more_btn)
            time.sleep(0.3)
            cleanup_popup(driver)

            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(SLEEP_CLICK_MORE)
        except:
            # 클릭 실패하면 한번 더 시도
            try:
                time.sleep(0.8)
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(SLEEP_CLICK_MORE)
            except:
                print("⚠️ 더보기 클릭 실패 → 종료")
                break

        # 링크 수 증가가 멈추면 안전 종료
        if len(links) == last_count:
            stagnant_round += 1
        else:
            stagnant_round = 0
            last_count = len(links)

        if stagnant_round >= 3:
            print("⚠️ 링크 증가 멈춤(3회 연속) → 종료")
            break

    return list(links)


# 3) 상세 페이지에서 혜택 긁기

def parse_detail_page(link: str):
    driver.get(link)
    cleanup_popup(driver)

    # 카드명 로딩 기다리기
    try:
        wait.until(lambda d: d.find_element(By.CSS_SELECTOR, "strong.card").get_attribute("textContent").strip() != "")
    except TimeoutException:
        return None  # 스킵

    # 카드명/카드사
    try:
        card_name = driver.find_element(By.CSS_SELECTOR, "strong.card").get_attribute("textContent").strip()
    except:
        card_name = ""
    try:
        card_company = driver.find_element(By.CSS_SELECTOR, "p.brand").get_attribute("textContent").strip()
    except:
        card_company = ""

    card_info = {
        "card_name": card_name,
        "card_company": card_company,
        "benefits": []
    }

    # 주요혜택 영역 로딩
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.cmd_con.benefit dl")))
    except TimeoutException:
        return card_info  # 혜택 못 읽어도 기본정보 저장

    dls = driver.find_elements(By.CSS_SELECTOR, "article.cmd_con.benefit dl")

    for dl in dls:
        cleanup_popup(driver)

        # 카테고리
        try:
            category = dl.find_element(By.CSS_SELECTOR, "dt p.txt1").get_attribute("textContent").strip()
        except:
            category = ""

        # 접혀있으면 펼치기(on 클래스 없으면 클릭)
        try:
            if "on" not in (dl.get_attribute("class") or ""):
                dt = dl.find_element(By.TAG_NAME, "dt")
                driver.execute_script("arguments[0].click();", dt)
                time.sleep(0.25)
        except:
            pass

        # 내용
        try:
            content = dl.find_element(By.TAG_NAME, "dd").get_attribute("textContent").strip()
        except:
            content = ""

        if category or content:
            card_info["benefits"].append({
                "category": category,
                "content": content
            })

    return card_info


# 4) 실행

result = []

try:
    print("=== 1) 링크 수집 시작 ===")
    links = collect_links_all()
    print(f"✅ 최종 링크 수집 개수: {len(links)}")

    print("\n=== 2) 상세 페이지 크롤링 시작 ===")
    for idx, link in enumerate(links, 1):
        info = parse_detail_page(link)
        if info is not None:
            result.append(info)

        print(f"[{idx}/{len(links)}] {info['card_name'] if info else '(스킵)'} 완료 | 누적 저장: {len(result)}")

        # 중간 저장
        if idx % SAVE_EVERY == 0:
            with open(OUT_PATH, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            print(f"💾 중간 저장 완료: {OUT_PATH} (누적 {len(result)}개)")

        time.sleep(SLEEP_BETWEEN_CARDS)

    # 최종 저장
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"\n✅ 최종 저장 완료: {OUT_PATH} (총 {len(result)}개)")

finally:
    driver.quit()

### 1-2. 체크카드

In [ ]:
LIST_URL = "https://www.card-gorilla.com/search/card?cate=CHK&annual=0,5&pre=0,3&sort=ranking&is_discon=1"
OUT_PATH = "cardgorilla__check_crd_374.json"

TARGET_N = 374            # 전체 카드 수(표시용). 실제 종료는 더보기 버튼 없어질 때까지.
SAVE_EVERY = 25           # 중간 저장 주기
SLEEP_CLICK_MORE = 1.2    # 더보기 클릭 후 대기
SLEEP_BETWEEN_CARDS = 0.5 # 카드 상세 페이지 간 대기

driver = wb.Chrome()
driver.implicitly_wait(3)
wait = WebDriverWait(driver, 12)


# 1) 팝업/새창 정리 유틸

def cleanup_popup(driver):
    # 새 창/탭 닫기
    if len(driver.window_handles) > 1:
        main = driver.window_handles[0]
        for h in driver.window_handles[1:]:
            driver.switch_to.window(h)
            driver.close()
        driver.switch_to.window(main)

    # alert 닫기
    try:
        driver.switch_to.alert.dismiss()
    except NoAlertPresentException:
        pass
    except:
        pass


# 2) 리스트에서 detail 링크 전부 수집

def collect_links_all():
    driver.get(LIST_URL)
    cleanup_popup(driver)

    # 카드 리스트가 로딩될 때까지
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.b_view[href^='/card/detail/']")))

    links = set()
    last_count = 0
    stagnant_round = 0

    while True:
        cleanup_popup(driver)

        # 현재 화면에 있는 detail 링크 수집
        elems = driver.find_elements(By.CSS_SELECTOR, "a.b_view[href^='/card/detail/']")
        for e in elems:
            href = e.get_attribute("href")
            if href:
                links.add(href)

        print(f"현재 링크 수: {len(links)} / {TARGET_N}(목표표시)")

        # 더보기 버튼 찾기 (없으면 종료)
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, "a.lst_more")
        except:
            print("  더보기 버튼 없음 → 링크 수집 종료")
            break

        # 더보기 버튼 클릭
        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", more_btn)
            time.sleep(0.3)
            cleanup_popup(driver)

            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(SLEEP_CLICK_MORE)
        except:
            # 클릭 실패하면 한번 더 시도
            try:
                time.sleep(0.8)
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(SLEEP_CLICK_MORE)
            except:
                print("  더보기 클릭 실패 → 종료")
                break

        # 링크 수 증가가 멈추면 안전 종료
        if len(links) == last_count:
            stagnant_round += 1
        else:
            stagnant_round = 0
            last_count = len(links)

        if stagnant_round >= 3:
            print("  링크 증가 멈춤(3회 연속) → 종료")
            break

    return list(links)


# 3) 상세 페이지에서 혜택 긁기

def parse_detail_page(link: str):
    driver.get(link)
    cleanup_popup(driver)

    # 카드명 로딩 기다리기
    try:
        wait.until(lambda d: d.find_element(By.CSS_SELECTOR, "strong.card").get_attribute("textContent").strip() != "")
    except TimeoutException:
        return None  # 스킵

    # 카드명/카드사
    try:
        card_name = driver.find_element(By.CSS_SELECTOR, "strong.card").get_attribute("textContent").strip()
    except:
        card_name = ""
    try:
        card_company = driver.find_element(By.CSS_SELECTOR, "p.brand").get_attribute("textContent").strip()
    except:
        card_company = ""

    card_info = {
        "card_name": card_name,
        "card_company": card_company,
        "benefits": []
    }

    # 주요혜택 영역 로딩
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.cmd_con.benefit dl")))
    except TimeoutException:
        return card_info  # 혜택 못 읽어도 기본정보 저장

    dls = driver.find_elements(By.CSS_SELECTOR, "article.cmd_con.benefit dl")

    for dl in dls:
        cleanup_popup(driver)

        # 카테고리
        try:
            category = dl.find_element(By.CSS_SELECTOR, "dt p.txt1").get_attribute("textContent").strip()
        except:
            category = ""

        # 접혀있으면 펼치기(on 클래스 없으면 클릭)
        try:
            if "on" not in (dl.get_attribute("class") or ""):
                dt = dl.find_element(By.TAG_NAME, "dt")
                driver.execute_script("arguments[0].click();", dt)
                time.sleep(0.25)
        except:
            pass

        # 내용
        try:
            content = dl.find_element(By.TAG_NAME, "dd").get_attribute("textContent").strip()
        except:
            content = ""

        if category or content:
            card_info["benefits"].append({
                "category": category,
                "content": content
            })

    return card_info


# 4) 실행

result = []

try:
    print("=== 1) 링크 수집 시작 ===")
    links = collect_links_all()
    print(f"  최종 링크 수집 개수: {len(links)}")

    print("\n=== 2) 상세 페이지 크롤링 시작 ===")
    for idx, link in enumerate(links, 1):
        info = parse_detail_page(link)
        if info is not None:
            result.append(info)

        print(f"[{idx}/{len(links)}] {info['card_name'] if info else '(스킵)'} 완료 | 누적 저장: {len(result)}")

        # 중간 저장
        if idx % SAVE_EVERY == 0:
            with open(OUT_PATH, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            print(f"  중간 저장 완료: {OUT_PATH} (누적 {len(result)}개)")

        time.sleep(SLEEP_BETWEEN_CARDS)

    # 최종 저장
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"\n  최종 저장 완료: {OUT_PATH} (총 {len(result)}개)")

finally:
    driver.quit()

## 1-3. 크롤링 결과

- 신용카드
806개 데이터 크롤링 완료
- 체크카드
374개 데이터 크롤링 완료

# 2. 페르소나 지정
## 상세 페르소나 설정 
- Case 1 : "취준생 김알뜰 씨"
    - persona : 대중교통이용, 편의점이용, 공과금, 통신비 할인 필요
    - question : 저는 매일 지하철로 학원을 다녀요. 점심은 주로 편의점 도시락으로 해결하고, 자취방 전기세랑 핸드폰 요금 나갈 때마다 손이 떨립니다. 최대한 고정 지출을 줄여줄 카드가 필요해요. 아, 전월 실적은 30만 원 정도가 한계예요.
    
- Case 2: "직장인 이여행 씨"
    - persona : 배달 앱 이용, OTT 구독, 온오프라인 쇼핑, 항공할인 필요
    - question : 퇴근 후 배달 음식을 시켜 먹으며 넷플릭스를 보는 게 유일한 낙이에요. 주말엔 무조건 무신사나 쿠팡에서 쇼핑을 하고, 1년에 두 번은 꼭 해외로 떠납니다. 실적은 신경안 써요, 혜택만 확실하면 됩니다!

- Case 3: "학부모 박마트 씨"
    - persona : 교육 할인 필요, 생활비, 주유 할인 필요, 공과금 및 교육 할인 필요
    - question : 아이들 학원비가 매달 100만 원 넘게 나가요. 주말에는 온 가족이 대형마트에서 장을 보고, 자차로 아이들을 등하교시킵니다. 아파트 관리비와 학원비에서 조금이라도 할인을 받고 싶어요.


In [2]:
question1 = """
저는 매일 지하철로 학원을 다녀요. 점심은 주로 편의점 도시락으로 해결하고, 자취
방 전기세랑 핸드폰 요금 나갈 때마다 손이 떨립니다. 최대한 고정 지출을 줄여줄 카
드가 필요해요. 아, 전월 실적은 30만 원 정도가 한계예요.
"""

question2 = """
퇴근 후 배달 음식을 시켜 먹으며 넷플릭스를 보는 게 유일한 낙이에요. 
주말엔 무조건 무신사나 쿠팡에서 쇼핑을 하고, 1년에 두 번은 꼭 해외로 떠납니다. 
실적은 신경안 써요, 혜택만 확실하면 됩니다!
"""

question3 = """
아이들 학원비가 매달 100만 원 넘게 나가요. 
주말에는 온 가족이 대형마트에서 장을 보고, 자차로 아이들을 등하교시킵니다. 
아파트 관리비와 학원비에서 조금이라도 할인을 받고 싶어요.
"""

# 3. Base 모델 구축
## 1) 신용카드

In [ ]:
from langchain_openai import ChatOpenAI

llm_base = ChatOpenAI(model="gpt-5-nano", temperature=0)

def run_base(question):
    prompt = f"""
다음 사용자에게 한국에서 발급 가능한 신용카드를 3개씩 추천하시오.
추천 이유를 반드시 포함하시오.

[사용자 질문]
{question}
"""
    return llm_base.invoke(prompt).content


# ===== Case 1 =====
print("===== 페르소나 1 =====")
print(run_base(question1))

# ===== Case 2 =====
print("\n===== 페르소나 2 =====")
print(run_base(question2))

# ===== Case 3 =====
print("\n===== 페르소나 3 =====")
print(run_base(question3))

cases = {
    "페르소나 1": question1,
    "페르소나 2": question2,
    "페르소나 3": question3
}

for case_name, question in cases.items():
    print("\n==============================")
    print(f"{case_name} (temperature=0, 10회 실행)")
    print("==============================")

    for i in range(10):
        print(f"\n----- 응답 {i+1} -----")
        print(run_base(question))


## 2) 체크카드

In [ ]:
from langchain_openai import ChatOpenAI

llm_base = ChatOpenAI(model="gpt-5-nano", temperature=0)

def run_base(question):
    prompt = f"""
다음 사용자에게 한국에서 발급 가능한 체크카드를 3개씩 추천하시오.
추천 이유를 반드시 포함하시오.

[사용자 질문]
{question}
"""
    return llm_base.invoke(prompt).content


# ===== Case 1 =====
print("===== 페르소나 1 =====")
print(run_base(question1))

# ===== Case 2 =====
print("\n===== 페르소나 2 =====")
print(run_base(question2))

# ===== Case 3 =====
print("\n===== 페르소나 3 =====")
print(run_base(question3))

cases = {
    "페르소나 1": question1,
    "페르소나 2": question2,
    "페르소나 3": question3
}

for case_name, question in cases.items():
    print("\n==============================")
    print(f"{case_name} (temperature=0, 10회 실행)")
    print("==============================")

    for i in range(10):
        print(f"\n----- 응답 {i+1} -----")
        print(run_base(question))

# 4. 모델 구현

In [ ]:
import os
import json

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

## 4-1. 신용카드 추천모델

In [9]:
with open("cardgorilla_crd_806_fee.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("총 카드 수:", len(data))

총 카드 수: 806


In [10]:
documents = []

for card in data:
    card_name = card["card_name"]
    card_company = card["card_company"]

    benefit_texts = []
    for b in card["benefits"]:
        category = b.get("category", "")
        content = b.get("content", "")
        benefit_texts.append(f"[{category}] {content}")

    full_text = f"""
카드사: {card_company}
카드명: {card_name}

혜택:
{chr(10).join(benefit_texts)}
"""

    documents.append(Document(page_content=full_text))

print("생성된 Document 수:", len(documents))

생성된 Document 수: 806


In [11]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(documents, embeddings)
print("벡터 개수:", vectorstore._collection.count())

벡터 개수: 806


In [12]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [ ]:
prompt1 = ChatPromptTemplate.from_template("""
너는 카드 추천 전문가다.
반드시 아래 Context에 포함된 정보만 근거로 판단하라.
Context에 없는 내용은 절대 추론하지 말고,
확인되지 않는 정보는 "문서에서 확인 불가"라고 명시하라.

──────────────────────────────
[추천 기준]
1. 사용자 요구와 가장 관련성이 높은 카드 3개를 선정한다.
2. Context에 없는 혜택은 절대 언급하지 않는다.
3. 혜택이 명확한 카드만 추천한다.
4. 동일한 카드를 중복해서 추천하지 않는다.
                                      
──────────────────────────────
[출력 형식]

━━━━━━━━━━━━━━━━━━
[카드 1]

:one: 카드 이름 / 카드사

:two: 주요 혜택 요약
- 사용자 소비패턴과 직접 연결되는 혜택만 정리

:three: 연회비 / 전월실적
- 연회비
- 전월실적

:four: 추천 이유
- 질문과의 연결성 중심 설명

:five: 주의사항
- 전월 실적, 할인 한도, 제외 업종 등 Context에 있는 경우만 작성

:six: 근거 문장 인용
- "Context 원문 일부"
━━━━━━━━━━━━━━━━━━

──────────────────────────────
[사용자 질문]
{question}

──────────────────────────────
[Context]
{context}
""")

## 4-2. 체크카드 추천모델

In [13]:
with open("cardgorilla__check_crd_374.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("총 카드 수:", len(data))

총 카드 수: 374


In [14]:
documents_2 = []

for card in data:
    card_name = card["card_name"]
    card_company = card["card_company"]

    benefit_texts = []
    for b in card["benefits"]:
        category = b.get("category", "")
        content = b.get("content", "")
        benefit_texts.append(f"[{category}] {content}")

    full_text = f"""
카드사: {card_company}
카드명: {card_name}

혜택:
{chr(10).join(benefit_texts)}
"""

    documents_2.append(Document(page_content=full_text))

print("생성된 Document 수:", len(documents_2))

생성된 Document 수: 374


In [ ]:
embeddings_2 = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore_2 = Chroma.from_documents(documents_2, embeddings_2)
print("벡터 개수:", vectorstore_2._collection.count())

In [16]:
retriever_2 = vectorstore_2.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [47]:
prompt2 = ChatPromptTemplate.from_template("""
너는 카드 추천 전문가다.
반드시 아래 Context에 포함된 정보만 근거로 판단하라.
Context에 없는 내용은 절대 추론하지 말고,
확인되지 않는 정보는 "문서에서 확인 불가"라고 명시하라.

──────────────────────────────
[추천 기준]
1. 사용자 요구와 가장 관련성이 높은 카드 3개를 선정한다.
1-2. 추천카드를 중복 출력하지 않는다.
2. Context에 없는 혜택은 절대 언급하지 않는다.
3. 혜택이 명확한 카드만 추천한다.


──────────────────────────────
[출력 형식]

━━━━━━━━━━━━━━━━━━
[카드 1]

1️⃣ 카드사 / 카드명

2️⃣ 주요 혜택 요약
- 사용자 소비패턴과 직접 연결되는 혜택만 정리

3️⃣ 추천 이유
- 질문과의 연결성 중심 설명

4️⃣ 주의사항
- 전월 실적, 할인 한도, 제외 업종 등 Context에 있는 경우만 작성

5️⃣ 근거 문장 인용
- "Context 원문 일부"
━━━━━━━━━━━━━━━━━━

──────────────────────────────
[사용자 질문]
{question}

──────────────────────────────
[Context]
{context}
""")

## 5-1-1. 신용카드 추천모델, temperature = 0

In [36]:
llm_credit_1 = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)

chain_credit_1 = prompt1 | llm_credit_1 | StrOutputParser()

In [37]:
def credit_cardbot_1(question: str):
    docs = retriever.invoke(question)

    context = "\n\n".join([d.page_content for d in docs])

    return chain_credit_1.invoke({
        "question": question,
        "context": context
    })

In [ ]:
import csv 
from datetime import datetime

def run_experiments_to_csv(
    questions,
    n_runs: int = 10,
    out_path: str = None
):
    if out_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = f"credit_runs_1{ts}.csv"

    # CSV 컬럼: 실행번호, 케이스명, 질문, 답변
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["run_idx", "case", "question", "answer"]
        )
        writer.writeheader()

        for run_idx in range(1, n_runs + 1):
            for case_name, q in questions:
                try:
                    ans = credit_cardbot_1(q)
                except Exception as e:
                    ans = f"[ERROR] {type(e).__name__}: {e}"

                writer.writerow({
                    "run_idx": run_idx,
                    "case": case_name,
                    "question": q,
                    "answer": ans
                })

    return out_path

In [ ]:
# ===== 실행 =====
questions = [
    ("persona_1", question1),
    ("persona_2", question2),
    ("persona_3", question3),
]

csv_path = run_experiments_to_csv(questions, n_runs=10)
print("Saved:", csv_path)

Saved: credit_runs_120260225_150606.csv


## 5-1-2. 체크카드 추천모델, temperature = 0

In [48]:
llm_check_1 = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)

chain_check_1 = prompt2 | llm_check_1 | StrOutputParser()

In [49]:
def check_cardbot_1(question: str):
    docs = retriever_2.invoke(question)

    context = "\n\n".join([d.page_content for d in docs])

    return chain_check_1.invoke({
        "question": question,
        "context": context
    })

In [50]:
import csv 
from datetime import datetime


def run_experiments_to_2csv(
    questions,
    n_runs: int = 1,
    out_path: str = None
):
    if out_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = f"check_runs_1{ts}.csv"

    # CSV 컬럼: 실행번호, 케이스명, 질문, 답변
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["run_idx", "case", "question", "answer"]
        )
        writer.writeheader()

        for run_idx in range(1, n_runs + 1):
            for case_name, q in questions:
                try:
                    ans = check_cardbot_1(q)
                except Exception as e:
                    ans = f"[ERROR] {type(e).__name__}: {e}"

                writer.writerow({
                    "run_idx": run_idx,
                    "case": case_name,
                    "question": q,
                    "answer": ans
                })

    return out_path

In [ ]:
# ===== 실행 =====
questions = [
    ("persona_1", question1),
    ("persona_2", question2),
    ("persona_3", question3),
]

csv_path = run_experiments_to_csv(questions, n_runs=1)
print("Saved:", csv_path)

# 5-1-3. Base 모델

## 페르소나 1 - 신용
좋은 카드로 고정지출을 줄이고 싶으시군요. 요청에 맞춰 한국에서 발급 가능한 신용카드 3개를 추천해 드리려면 몇 가지 정보를 더 확인하면 더 정확하게 맞춤 추천이 가능합니다.

확인하고 싶은 점
- 연회비 선호 여부: 무연회비가 좋을지, 연회비가 있어도 혜택이 많으면 괜찮은지요?
- 혜택 성향 선호: 현금처럼 쓰이는 포인트/캐시백 중 어떤 형태를 선호하시나요?(예: 현금화가 쉬운 캐시백 vs 가맹점 제휴사에서 쌓이는 포인트)
- 지출 구성 비중: 지하철/대중교통, 편의점(주로 어떤 편의점 체인 이용 여부도 포함), 공과금(전기세, 핸드폰 요금) 중에서 가장 큰 비중을 차지하는 항목은 무엇인가요? 예를 들어 공과금/통신요금 자동결제 할인에 초점을 맞추길 원하시는지요?
- 전월실적 목표: “전월 30만 원 정도”라는 한계를 감안해도 충분히 혜택을 받으려면 어떤 기준으로 혜택을 보시길 원하나요? 예를 들어 30만원 중 몇 %를 특정 혜택으로 최대화하는 걸 원하시는지요?
- 카드 발급 가능 여부/지역: 국내에서 발급 가능한지, 거주지에 따라 신청 가능한 카드가 다를 수 있습니다. 혹시 몇 군데 카드사 선호가 있나요?

참고로, 아래는 상황에 맞춰 자주 추천되는 카드 유형의 예시입니다. 이 유형으로 실제 카드 3종을 바로 드리려면 최신 혜택이 변동될 수 있어 발급사 공식 페이지로 확인이 필요합니다. 원하시면 제가 최신 정보로 실제 카드 3종을 바로 찾아 드리겠습니다.

- 유형 A: 무연회비로 시작하고 지하철/대중교통 + 편의점 할인에 강한 카드
  - 장점: 매일 사용하는 지출(교통/편의점)에서 안정적인 혜택을 받기 쉬움, 연회비 부담 없음
  - 주의점: 편의점/교통의 특정 제휴처에 한정된 혜택이 많아, 다른 지출이 많은 경우 추가 혜택을 보완해야 할 수도 있음

- 유형 B: 편의점 할인 집중형 카드
  - 장점: 점심 도시락 같은 편의점 지출에서 비교적 높은 할인 또는 캐시백
  - 주의점: 다른 일반 지출에서의 수익이 상대적으로 낮아지기 쉬움

- 유형 C: 공과금/통신요금 자동결제 혜택이 강한 카드
  - 장점: 매달 공과금·통신요금 자동결제로 일정 혜택 유지가 가능
  - 주의점: 전월 실적 산정 방식에 따라 혜택이 달라질 수 있음(특정 결제 제외 등)

다음 단계 제안
- 위의 정보를 조금만 알려주시면, 귀하의 요구에 맞춘 실제 카드 3종을 최신 정보로 정확하게 추천해 드리겠습니다. 
- 원하시면 지금 바로 제가 최신 카드 혜택을 확인해 3종을 구체적으로 제시해 드리겠습니다. 

괜찮으시다면, 선호하는 연회비 여부와 혜택 형태(현금화 vs 포인트), 그리고 1개월 지출에서 가장 중점을 두고 싶은 항목을 간단히 답해 주세요. 그러면 바로 3종 카드 후보를 구체적으로 제시해 드리겠습니다.

## 페르소나 1 - 체크
요청하신 대로 한국에서 발급 가능한 체크카드 중, 매일 지하철 이용·편의점 점심 소비·전기세/휴대폰 요금 같은 고정 지출이 큰 사용 패턴에 맞춰 전월 실적 30만원대에서도 혜택을 최대화할 수 있도록 3가지 유형의 추천안을 드립니다. 다만 카드 혜택은 카드사 정책에 따라 자주 변경되니, 실제 발급 시점에 혜택 여부를 꼭 확인해 주세요.

추가 정보 없이 바로 확정 3장을 내놓기보다는 아래의 3가지 유형으로 구성해 보았으며, 원하시면 이 중에서 실제 발급 가능한 구체 카드 이름으로 3장을 확정해 드리겠습니다.

1) 유형 A: 지하철/대중교통 + 편의점 중심의 다목적 체크카드
- 추천 포커스
  - 지하철·버스 같은 대중교통 할인 혜택
  - 편의점 체인(CU/GS25/7-Eleven 등) 할인 또는 제휴 혜택
  - 기본 결제 건당 사용이 쉬워 전월 실적 30만원대에서도 혜택이 누적되기 쉬움
- 왜 이 카드가 지금 필요에 맞나
  - 매일 지하철 이용이 핵심이므로 교통 혜택이 큰 카드를 선택하면 운전/출퇴근 비용 절감에 직접적 도움
  - 점심은 주로 편의점 도시락이므로 편의점 할인으로 식비 절감 가능
  - 연회비가 거의 없거나 낮은 편이 많아 정기적으로 부담 없이 사용할 수 있음
- 사용 팁
  - 교통비와 편의점 결제를 이 카드로 일관되게 우선 결제
  - 전월 실적 산정 시 교통/편의점 결제가 실적에 포함되는지 확인하고, 필요 시 온라인 결제나 공과금 자동이체를 보너스 결제로 활용

2) 유형 B: 통신비/온라인 결제 중심의 절감형 체크카드
- 추천 포커스
  - 휴대폰 요금 할인 또는 캐시백
  - 온라인 결제 또는 구독형 결제에 강한 혜택(인터넷/콘텐츠 구독, 배달앱 등)
  - 편의점 할인이나 마트 할인까지 겸하는 경우도 있음
- 왜 이 카드가 지금 필요나
  - 매월 핸드폰 요금이 큰 편이라 통신비 할인으로 고정지출을 직접 줄일 수 있음
  - 편의점 식사나 소액 결제도 가능하므로 전월 실적 30만원대에서 혜택을 누릴 여지가 큼
- 사용 팁
  - 핸드폰 요금 자동 납부를 이 카드로 설정해 할인/캐시백을 꾸준히 확보
  - 온라인 쇼핑/필수 결제(공과금, 구독료 등)도 이 카드로 집중 결제하여 실적 관리

3) 유형 C: 공과금/에너지 비용 절감형 체크카드
- 추천 포커스
  - 전기세, 수도세 등 공과금 할인 또는 캐시백
  - 편의점/마트 할인과 더불어 소액 지출에서도 혜택 체감 가능
  - 연회비가 낮거나 없고, 전월 실적 조건이 비교적 완화된 편인 경우가 많음
- 왜 이 카드가 지금 필요나
  - 자취 생활에서 전기세 등 고정비가 큰 만큼, 공과금 할인이 실제 지출 절감에 큰 효과
  - 지출의 큰 축이 공과금이 되지 않는 경우에도 편의점·마트 혜택으로 전월 실적 확보에 도움
- 사용 팁
  - 전기세/수도세 등 공과금 납부를 이 카드로 자동 납부
  - 편의점/마트 결제도 병행하여 실적 충족과 함께 혜택 최적화

실제 3개 카드를 확정해 드리려면 아래 정보를 알려 주시면 더 정확하게 맞춤 제안을 드리겠습니다.
- 선호하는 은행/카드사(예: 국민은행, 신한은행, 하나은행, 우리은행, NH농협 등)나 특정 브랜드 선호 여부
- 연회비 여부에 대한 선호(무연회비를 기본으로 하고, 필요 시에만 연회비가 있는 상품 선호 등)
- 지하철/편의점/통신비 중 어떤 혜택을 더 우선시하시는지
- 월 실적의 활용 계획(전월 30만원 한계 내에서 어떤 지출을 집중적으로 늘릴지에 대한 아이디어)

# 5-1-4. 신용카드 모델 tem=0
[카드 1]

1️⃣ BNK부산은행 오늘은e 신용카드

2️⃣ 주요 혜택 요약
- 대중교통 할인: 버스, 지하철, 택시요금 5% 청구할인 - 월 최대 5천원 할인
- 간편결제 할인: PAYCO, 삼성페이, 네이버페이, 카카오페이, 썸패스 등 간편결제 이용 시 5% 청구할인 - 월 최대 1만원 할인
- 편의점/카페/학원/통신 등 업종 할인: 편의점 5% 청구할인(월 최대 5천원), 카페 5% 청구할인(월 최대 5천원), 학원 5% 청구할인(월 최대 5천원), 이동통신 요금 5% 청구할인(월 최대 5천원)

3️⃣ 연회비 / 전월실적
- 연회비: 7,000원
- 전월실적: 전월 실적 30만원 이상 시 해당 서비스 제공

4️⃣ 추천 이유
- 매일 지하철로 학원을 다니고 편의점 도시락 소비가 많으며, 핸드폰 요금 등 고정 지출이 큰 사용패턴과 직접적으로 연결되는 할인들(대중교통, 편의점/카페, 통신, 간편결제)이 존재합니다. 전월실적 요건도 30만원 이상으로 현재 사용패턴과 맞아 혜택 적용 가능성이 높습니다.

5️⃣ 주의사항
- 전월 실적 30만원 이상 시 서비스 제공. 간편결제 할인 및 생활 할인은 카드 수령일로부터 다음달 말일까지 전월 실적 미달 시에도 제공될 수 있음(다만 기본 요건은 30만원 이상 시 서비스 제공으로 명시).
- 전월실적 제외 업종: 제세공과금, 전기요금, 도시가스요금, 대학등록금 등 등 일부 공과금은 전월실적 산정에서 제외될 수 있습니다.
- 무이자할부 이용 시 할인 혜택은 적용되지 않습니다.

6️⃣ 근거 문장 인용
- "대중교통 할인 서비스- 버스, 지하철, 택시요금 5% 청구할인- 월 최대 5천원 할인- 전월실적 30만원 이상 시 서비스 제공"
- "간편결제 할인 서비스- PAYCO, 삼성페이, 네이버페이, 카카오페이, 썸패스 5% 청구인 할인- 월 최대 1만원 할인- 전월 실적 30만원 이상 시 서비스 제공- 1만원 이상 결제시 제공"
- "- 전월실적 제외 업종: 제세공과금(국세,지방세,4대보험료 등), 전기요금, 도시가스요금, 대학등록금, 상품권 등"

# 5-1-5. 체크카드 모델 tem=0
[카드 1]

1️⃣ 트레이더스신세계 삼성카드 / 삼성카드

2️⃣ 주요 혜택 요약
- 대중교통 10% 청구할인 (버스/지하철) - 전월 이용금액이 2만원 이상 시 적용
- 이동통신 5% 청구할인
- 커피전문점(카페) 5% 결제일할인(청구할인)

3️⃣ 연회비 / 전월실적
- 연회비: Context 원문에서 확인 불가
- 전월실적
- 대중교통 혜택은 전월 이용금액이 2만원 이상 시 적용

4️⃣ 추천 이유
- 지하철 등 평소 출퇴근으로 소모가 큰 대중교통 비용을 크게 절감할 수 있으며, 카페/통신 혜택이 함께 있어 매일의 고정지출을 다각도로 줄이는 데 유리합니다.

5️⃣ 주의사항
- 할인 혜택은 승인 시점 기준으로 적용되며, 삼성카드의 다른 결제일할인과 중복 적용되지 않음
- 해외 결제건/무승인 결제건의 경우 매출전표 접수 시점 기준 적용
- 전월 실적 산정 시 특정 가맹점 이용액은 제외될 수 있으며, 할인 혜택이 큰 금액만 적용될 수 있음
- 발급월에는 특정 구간의 할인율/한도가 다르게 적용될 수 있음
- 월별 할인한도는 각 혜택별로 다르게 설정될 수 있음

6️⃣ 근거 문장 인용
- "대중교통] 버스/지하철 10% 청구할인- 각 교통사업자별 월 이용금액이 2만원 이상 시"
- "카페] 커피전문점 5% 결제일할인(청구할인)대상점"
- "통신] 이동통신 5% 결제일할인(청구할인)" 
- "할인 혜택은 승인 시점 기준으로 적용됩니다. 다만, 해외 결제건 및 무승인 결제건…의 경우 매출전표 접수 시점 기준으로 적용됩니다."
- "전월 이용금액이란, 매월 1일부터 말일까지 이용한 일시불 및 할부 이용금액을 의미합니다."

## 5-2-1. 신용카드 추천모델, teamperature 0.8

In [33]:
llm_credit_2 = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0.8
)

chain_credit_2 = prompt1 | llm_credit_2 | StrOutputParser()

In [34]:
def credit_cardbot_2(question: str):
    docs = retriever.invoke(question)

    context = "\n\n".join([d.page_content for d in docs])

    return chain_credit_2.invoke({
        "question": question,
        "context": context
    })

In [ ]:
import csv 
from datetime import datetime

def run_experiments_to3_csv(
    questions,
    n_runs: int = 10,
    out_path: str = None
):
    if out_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = f"credit_runs_2{ts}.csv"

    # CSV 컬럼: 실행번호, 케이스명, 질문, 답변
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["run_idx", "case", "question", "answer"]
        )
        writer.writeheader()

        for run_idx in range(1, n_runs + 1):
            for case_name, q in questions:
                try:
                    ans = credit_cardbot_2(q)
                except Exception as e:
                    ans = f"[ERROR] {type(e).__name__}: {e}"

                writer.writerow({
                    "run_idx": run_idx,
                    "case": case_name,
                    "question": q,
                    "answer": ans
                })

    return out_path

## 5-2-2. 체크카드 추천모델, temperature = 0.8

In [29]:
llm_check_2 = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0.8
)

chain_check_2 = prompt2 | llm_check_2 | StrOutputParser()

In [30]:
def check_cardbot_2(question: str):
    docs = retriever_2.invoke(question)

    context = "\n\n".join([d.page_content for d in docs])

    return chain_check_2.invoke({
        "question": question,
        "context": context
    })

In [ ]:
import csv 
from datetime import datetime

def run_experiments_to4_csv(
    questions,
    n_runs: int = 10,
    out_path: str = None
):
    if out_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = f"credit_runs_4{ts}.csv"

    # CSV 컬럼: 실행번호, 케이스명, 질문, 답변
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["run_idx", "case", "question", "answer"]
        )
        writer.writeheader()

        for run_idx in range(1, n_runs + 1):
            for case_name, q in questions:
                try:
                    ans = check_cardbot_2(q)
                except Exception as e:
                    ans = f"[ERROR] {type(e).__name__}: {e}"

                writer.writerow({
                    "run_idx": run_idx,
                    "case": case_name,
                    "question": q,
                    "answer": ans
                })

    return out_path

# 6. 상, 중, 하 평가

## 신용카드

### 페르소나 1
    총 10회 중, 상 5개, 중 5개

### 페르소나 2
    총 10회 중, 상 4개, 중 5개, 하 1개

### 페르소나 3
    총 10회 중, 상 5개, 중 5개

### 하 수준 답변 예시
페르소나 2 : ott, 배달음식, 온라인쇼핑, 해외여행

답변 :
1️⃣ NC다이노스카드 / NH농협카드

2️⃣ 주요 혜택 요약
- Online Shopping: 온라인쇼핑 청구할인 (월 5천원~2만원) 10% 할인
  - 30만원 이상 시 월 할인한도 5천원
  - 50만원 이상 시 1만원
  - 100만원 이상 시 2만원
  - 150만원 이상 시 3만원
  - 대상 쇼핑: G마켓, 옥션, 인터파크, 11번가, 티켓링크(스포츠), 농협a마켓 등
- 영화/스포츠 관련 혜택도 있지만, 주된 연결은 온라인쇼핑 할인(패턴과의 연결성 강조)

3️⃣ 연회비 / 전월실적
- 연회비: 8,000
- 전월실적: - 온라인쇼핑 할인은 월별 한도와 금액 기준으로 적용(30/50/100/150만원 구간별)

4️⃣ 추천 이유
- 주말 온라인 쇼핑을 많이 하는 패턴에 10% 할인이라는 높은 할인율로 직접 연결될 수 있음.
- NC다이노스 홈구장 할인 등 엔터테인먼트 혜택도 있지만, 사용 패턴상 온라인쇼핑 혜택이 가장 직접적임.

5️⃣ 주의사항
- 해외가맹점 이용 시 카드서비스가 제공되지 않음
- 온라인쇼핑 할인은 특정 가맹점에 한정될 수 있으며, 전월실적 구간에 따라 한도가 달라짐
- 가맹점 분류상 해당 업종 이용액에 한해 적용

6️⃣ 근거 문장 인용
- ""[온라인쇼핑] 온라인쇼핑 청구할인 (월 5천원~2만원) - 할인율 10%""
- ""30만원 이상 : 월 할인한도 5천원; 50만원 이상 : 1만원; 100만원 이상 : 2만원; 150만원 이상 : 3만원""
- ""영화/스포츠 관련 혜택도 있지만, 해외가맹점 이용 시 카드서비스가 제공되지 않음"""


### 상 수준 답변 예시
페르소나1 : 대중교통, 편의점, 휴대전화

1️⃣  NH20 해봄카드 / NH농협카드

2️⃣ 주요 혜택 요약
- [대중교통] 대중교통(버스/지하철) 10% 청구할인- 각 교통사업자별 월 이용금액이 2만원 이상 시- 후불교통 이용건이며, 시외 · 고속버스는 제외됨
- [카페/디저트] 커피 30% 청구할인- 스타벅스, 이디야- 건당 이용금액 1만원 이상 시(월 최대 5,000원 할인)
- [편의점] GS25 5% 청구할인- 건당 이용금액 1만원 이상 시
- [통신] 이동통신 5% 청구할인- 월 최대 5천원 할인- 전월실적 시 서비스 제공 조건 포함
- [온라인쇼핑] 온라인 쇼핑몰 10% 청구할인- G마켓, 옥션, 인터파크, 11번가, 농협몰
- [도서] 온라인 서점, 어학시험 10% 청구할인- 온라인 서점/어학시험(토익 등) 건당 이용금액 2만원 이상 시

3️⃣ 연회비 / 전월실적
- 연회비: 8,000원
- 전월실적: 혜택별로 다름(예: 대중교통 10% 혜택은 전월실적 요구 조건이 명시되어 있음. 일부 혜택은 30만원 이상 시 제공)

4️⃣ 추천 이유
- 지하철 이용이 잦고, 편의점 식사도 자주 한다면 대중교통 10% 할인과 편의점 5% 할인 등 고정 지출에 직접 연결되는 혜택이 강합니다. 또한 커피 할인과 어학/도서 관련 혜택도 있어 학원/공부 관련 지출이 많을 때도 추가 이익이 생깁니다.

5️⃣ 주의사항
- 혜택별 전월실적 및 이용조건이 상이하므로 각 혜택의 조건을 확인 필요
- 일부 혜택은 특정 매장/업종에서만 적용
- 무이자할부 시 할인 혜택 미적용 가능

6️⃣ 근거 문장 인용
- ""[대중교통] 대중교통(버스/지하철) 10% 청구할인- 각 교통사업자별 월 이용금액이 2만원 이상 시- 후불교통 이용건이며, 시외 · 고속버스는 제외됨""
- ""[카페/디저트] 커피 30% 청구할인- 스타벅스, 이디야- 건당 이용금액 1만원 이상 시 (월 최대 5,000원 할인)""
- ""[편의점] GS25 5% 청구할인- 건당 이용금액 1만원 이상 시""
- ""[통신] 이동통신 5% 청구할인- 월 최대 5천원 할인""
- ""[온라인쇼핑] 온라인 쇼핑몰 10% 청구할인- G마켓, 옥션, 인터파크, 11번가, 농협몰""
- ""[도서] 온라인 서점, 어학시험 10% 청구할인- 온라인 서점: YES24, 인터파크도서, 알라딘, 교보문고, 반디앤루니스, 영풍문고""
2조_미니프로젝트에 메시지 보내기

## 체크카드

### 페르소나 1
    총 10회 중, 중 8개,하 2개,

### 페르소나 2
    총 10회 중, 상 4개, 중 6개

### 페르소나 3
    총 10회 중, 상 9개, 중 1개

### 답변 예시

### 하 수준 답변 예시
페르소나 1 : 대중교통이용, 편의점이용, 공과금, 통신비 할인 필요

1️⃣ 케이뱅크 / 케이뱅크 플러스 체크카드

2️⃣ 주요 혜택 요약
[모든가맹점] 모든 가맹점에서 0.3% 캐시백 - 실적 조건 및 캐시백 한도 없이 모든 가맹점에서 0.3% 캐시백
캐시백률은 결제 건당 적용되며 1원 미만은 절사
캐시백 지급은 매출 접수일별로 1일 1회 결제계좌로 입금
단 국내 가맹점 이용금액 중 대학등록금, 아파트관리비, 상품권 및 기프트카드 결제금액, 각종 수수료, 이자, 제세공과금, 사회보험료, 교통카드 이용금액은 제외
[특별 캐시백] 최대 2만원 3% 특별 캐시백
월 한도: 전월 실적에 따라 1만원(30만원 이상), 2만원(50만원 이상)
신규 발급 시 월 1만원 혜택 제공(페이북 등록 등 이용실적 조건 없이)
[금융] 모든 은행, GS25 ATM 수수료 무료
[유의사항] 교통카드 이용금액은 캐시백 대상에서 제외되며, 전월 실적 산정 시 특정 공과금/등록금/요금/수수료 등은 제외

3️⃣ 추천 이유
기본적으로 모든 가맹점에서 0.3% 캐시백이 무조건 적용되며, 추가로 매달 특정 구간에 따라 1만~2만원의 특별 캐시백이 가능하므로 고정지출이 많은 분께 유리합니다. 다만 교통카드 이용은 캐시백 대상에서 제외되므로 대중교통 비용 절감과는 직접적으로 연결되진 않습니다.

4️⃣ 주의사항
교통카드 이용금액은 캐시백 대상에서 제외.
전월 실적 산정 시 대학등록금, 아파트관리비, 상품권/선불카드 구입 및 충전, 각종 수수료 및 이자 등 특정 항목 제외.
신규 발급 시 월 1만원 혜택은 조건 없이 제공되며, 이후 월 혜택은 전월 실적에 따라 차등 적용.

5️⃣ 근거 문장 인용
"[모든 가맹점] 모든 가맹점에서 0.3% 캐시백- 실적 조건 및 캐시백 한도 없이 모든 가맹점에서 0.3% 캐시백유의사항- 실적 조건 및 캐시백 한도 없이 국내외 모든 가맹점에서 0.3% 캐시백이 지급됩니다.- 캐시백률은 결제 건당 적용되며 1원 미만은 절사됩니다.- 캐시백 지급은 당행 매출 접수일별로 금액이 합산되어 1일 1회 결제계좌로 입금됩니다."
"[특별 캐시백] 최대 2만원 3% 특별 캐시백- 건당 금액/횟수 조건 없이 월 한도 내 자유롭게- 사용처 예: 모바일쇼핑/앱스토어/배달앱 등- 전월 실적 30만원 이상: 월 1만원- 전월 실적 50만원 이상: 월 2만원"
"[유의사항] 전월 실적 산정시 대학등록금, 아파트관리비, 상품권 및 기프트카드 결제금액, 각종 수수료, 이자, 제세공과금, 사회보험료, 교통카드 이용금액은 제외됩니다." 

### 상 수준 답변 예시
페르소나 3 : 교육 할인 필요, 마트, 주유 할인 필요, 공과금

1️⃣ MG 새마을금고 / 라이프핏 체크카드

2️⃣ 주요 혜택 요약
[공과금] 도시가스, 전기요금, 아파트관리비, 사립유치원 교육경비, 초중고 학부모부담금 자동납부 결제 시 5% 할인 캐시백 — 월 1회 한도 / 통합할인한도 적용
[통신] 통신비 대상가맹점 자동납부 결제 시 5% 할인 캐시백
[카페] 스타벅스, 커피빈, 투썸플레이스, 메가커피, 컴포즈커피, 빽다방, 매머드커피 결제 시(앱결제포함) 5% 할인 캐시백 — 일 1회, 월 5회 한도
[쇼핑] 대상가맹점 결제 시 1천원 할인캐시백 (오프라인 대형마트: 이마트, 롯데마트, 홈플러스 등 / 온라인 이커머스: 쿠팡, 마켓컬리, SSG.COM 등)
[주유] 전국 주유소 및 충전소, 전기차충전소 결제 시 1천원 할인캐시백 — 일 1회, 월 5회 한도

3️⃣ 추천 이유
아이들 학원비 할인과 아파트관리비 할인이라는 질문의 핵심 요구에 직접 대응하는 카드입니다. 학원 관련 혜택은 학원 자동 납부 관련 5% 할인과, 아파트관리비 포함 공과금 계열에 대한 5% 할인으로 구성되어 있어 월간 대규모 학원비 및 관리비 절감에 실질적 기여가 큽니다. 또한 주말 대형마트 쇼핑과 차량 운행 시 주유 혜택도 있어 거주 가구의 전반적 지출 절감에 도움이 됩니다.

4️⃣ 주의사항
전월 이용실적 50만원 이상 시 할인캐시백 제공(전표매입 기준); 세부 한도 및 제외 업종은 유의 필요.
학원 할인은 건당 5만원 이상 이용 시 적용, 일 1회/월 5회 한도.
아파트관리비, 공과금은 전월실적 산정 및 할인 적용에 영향을 주는 요소로, 일부 항목은 제외될 수 있습니다.
할인캐시백은 전표매입 후 입금되며, 취소 시 차감될 수 있습니다.
간편결제(카카오페이, 네이버페이 등)로 결제한 경우 할인캐시백이 적용되지 않을 수 있습니다.

5️⃣ 근거 문장 인용
"[공과금] 도시가스, 전기요금, 아파트가관리비, 사립유치원 교육경비, 초중고 학부모부담금 자동납부 결제 시 5% 할인 캐시백 - 월 1회 한도 / 통합할인한도 적용"
"[통신] 통신비 대상가맹점 자동납부 결제 시 5% 할인 캐시백"
"[카페] 스타벅스, 커피빈, 투썸플레이스, 메가커피, 컴포즈커피, 빽다방, 매머드커피 결제 시(앱결제포함) 5% 할인 캐시백 - 일 1회, 월 5회 한도"

"[쇼핑] 대상가맹점 결제 시 1천원 할인캐시백 … 오프라인 대형마트 이마트, 롯데마트, 홈플러스 … 온라인 이커머스 쿠팡, 마켓컬리, SSG.COM"
